# 01. 하네스 기초: 긴 상태를 루트 프롬프트에서 분리하기

목표: 긴 상태를 그대로 프롬프트에 붙이는 방식과, 상태를 변수로 오프로딩하는 하네스 방식을 비교합니다.

실행 방법: 위에서부터 순서대로 셀을 실행합니다. 외부 패키지는 필요하지 않습니다.

## 1. 같은 구조, 다른 도메인의 긴 상태 만들기

두 데이터셋은 표면 단어가 다르지만 구조는 같습니다. 하나는 주문 데이터, 다른 하나는 지원 티켓 데이터입니다. 일반 프롬프트는 전체 상태를 루트 모델에 직접 보여 주고, 하네스 프롬프트는 `DATA`와 `QUERY` 같은 변수만 보여 줍니다.

In [ ]:
commerce_state = "\n".join(
    f"order_{i:03d}: customer=segment_{i % 4}, status={'late' if i % 5 == 0 else 'ok'}, amount={20 + i}"
    for i in range(40)
)

support_state = "\n".join(
    f"ticket_{i:03d}: queue=team_{i % 3}, severity={'high' if i % 7 == 0 else 'normal'}, minutes={5 + i}"
    for i in range(40)
)

commerce_query = "status가 late인 주문 수와 amount 합계를 구하라."
support_query = "severity가 high인 티켓 수와 minutes 합계를 구하라."


def standard_prompt(state, query):
    # 일반 에이전트는 긴 상태와 질문을 그대로 하나의 루트 프롬프트에 붙이는 경우가 많습니다.
    return f"전체 상태:\n{state}\n\n질문:\n{query}\n\n모든 정보를 직접 읽고 답하라."


def harness_root_prompt(data_var="DATA", query_var="QUERY"):
    # 좋은 하네스는 루트 모델이 도메인별 원문보다 분해 절차를 보게 만듭니다.
    return "\n".join(
        [
            "You are the root policy of a data-analysis harness.",
            f"The large dataset is stored in variable {data_var}.",
            f"The user request is stored in variable {query_var}.",
            "Plan a small sequence of calls: select relevant records, aggregate fields, then format the answer.",
            "Do not inline the dataset into the root context.",
        ]
    )


standard_commerce = standard_prompt(commerce_state, commerce_query)
standard_support = standard_prompt(support_state, support_query)
harness_commerce = harness_root_prompt()
harness_support = harness_root_prompt()

for name, prompt in [
    ("standard_commerce", standard_commerce),
    ("standard_support", standard_support),
    ("harness_commerce", harness_commerce),
    ("harness_support", harness_support),
]:
    print(f"{name:18s} | characters={len(prompt):4d} | lines={prompt.count(chr(10)) + 1:3d}")

## 2. 루트 프롬프트 유사도 비교

원문의 주장처럼 하네스가 잘 설계되면 서로 다른 과제가 루트 모델에게 비슷한 관찰로 보입니다. 여기서는 단순한 3-gram Jaccard 유사도로 그 효과를 관찰합니다.

In [ ]:
import re


def tokenize(text):
    return re.findall(r"[A-Za-z0-9_가-힣]+", text.lower())


def ngrams(tokens, n=3):
    return set(tuple(tokens[i : i + n]) for i in range(max(0, len(tokens) - n + 1)))


def jaccard_ngram_similarity(a, b, n=3):
    a_grams = ngrams(tokenize(a), n)
    b_grams = ngrams(tokenize(b), n)
    if not a_grams and not b_grams:
        return 1.0
    return len(a_grams & b_grams) / len(a_grams | b_grams)


print("standard prompts similarity:", round(jaccard_ngram_similarity(standard_commerce, standard_support), 3))
print("harness prompts similarity: ", round(jaccard_ngram_similarity(harness_commerce, harness_support), 3))

## 3. 관찰

일반 프롬프트는 도메인별 레코드가 전부 들어가므로 두 작업의 표면 토큰이 크게 달라집니다. 반면 하네스 루트 프롬프트는 같은 제어 흐름을 유지합니다. 실제 RLM에서는 루트 모델이 이런 구조적 유사성을 학습하고, 도메인별 세부사항은 서브 호출이 처리하도록 만듭니다.